# AI-Based Motorcycle Helmet Detection and Rider Safety Monitoring System
## Model Training Pipeline using YOLOv8 & PyTorch on Google Colab (GPU)

This notebook trains a custom **YOLOv8** deep learning model to detect:
1. **Helmet** (Compliant Rider - Green)
2. **Without Helmet** (Traffic Safety Violation - Red)
3. **Motorcycle** (Two-Wheeler)
4. **Rider** (Motorcyclist / Person)

The trained model weights (`best.pt`) can be downloaded at the end of this notebook and plugged directly into your Streamlit project folder (`models/helmet_yolov8n.pt`).

---

### Step 1: Verify Hardware & GPU Acceleration
Ensure your Google Colab runtime is set to GPU: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# Check CUDA GPU availability
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")

### Step 2: Install Ultralytics YOLOv8 & Dependencies

In [ ]:
# Install latest Ultralytics and Roboflow dataset helpers
!pip install -q ultralytics roboflow opencv-python-headless matplotlib

### Step 3: Dataset Preparation & Download
You can acquire a verified Motorcycle Helmet dataset in YOLOv8 format via:
- **Roboflow Universe** (e.g. `helmet-detection-yolov8` or motorcycle safety datasets)
- **Kaggle** (Hardhat / Motorcycle Helmet Dataset)

The code below downloads an open-source motorcycle helmet dataset pre-annotated in YOLO format.

In [ ]:
import os
from pathlib import Path

# Method A: Direct download of public benchmark dataset in YOLO format
!mkdir -p /content/dataset
%cd /content/dataset

# Download a verified motorcycle helmet detection dataset archive
!wget -q --show-progress https://github.com/ultralytics/assets/releases/download/v0.0.0/motorcycle-helmet.zip -O helmet_data.zip || \
 curl -L "https://universe.roboflow.com/ds/your_token" -o helmet_data.zip

# If using Roboflow API, you can download directly via:
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("traffic-safety").project("motorcycle-helmet-detection")
# dataset = project.version(1).download("yolov8")

### Step 4: Configure `data.yaml` Dataset Descriptor
This YAML file defines dataset split paths (`train`, `val`, `test`), number of classes (`nc`), and class labels (`names`).

In [ ]:
data_yaml_content = """
path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 4
names: ['Helmet', 'Without Helmet', 'Motorcycle', 'Rider']
"""

with open('/content/dataset/data.yaml', 'w') as f:
    f.write(data_yaml_content.strip())

print("data.yaml configured successfully:")
!cat /content/dataset/data.yaml

### Step 5: Initialize YOLOv8 Model Architecture
We load pre-trained weights (`yolov8n.pt` or `yolov8s.pt`) which provide transferred visual features (edges, contours, objects) learned from ImageNet/COCO, allowing fast fine-tuning on helmet and motorcycle classes.

In [ ]:
from ultralytics import YOLO

# Load lightweight YOLOv8 nano model (optimized for real-time CPU & GPU inference)
model = YOLO('yolov8n.pt')

# Display model summary
print("Model architecture initialized:")
print(f"Classes configured: {model.names}")

### Step 6: Train the Model
Hyperparameters:
- **imgsz:** 640 (standard multi-scale resolution)
- **epochs:** 50 (or 100 for maximum mAP)
- **batch:** 16
- **optimizer:** AdamW or SGD
- **augmentations:** Mosaic, MixUp, HSV jitter, horizontal flips

In [ ]:
# Start training on GPU
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    device=0,
    workers=4,
    project='helmet_safety_run',
    name='yolov8n_helmet',
    save=True,
    plots=True,
    mosaic=1.0,
    mixup=0.1
)

### Step 7: Evaluate Training Metrics & Loss Curves
Ultralytics automatically generates:
- `results.png`: Precision, Recall, mAP@50, mAP@50-95, Box Loss, Class Loss, DFL Loss.
- `confusion_matrix.png`: Normalized confusion matrix showing classification accuracy per class.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# Display training loss & mAP metric curves
results_img = glob.glob('helmet_safety_run/yolov8n_helmet/results.png')
if results_img:
    plt.figure(figsize=(14, 8))
    plt.imshow(mpimg.imread(results_img[0]))
    plt.axis('off')
    plt.title('YOLOv8 Training Curves & mAP Progression', fontsize=14)
    plt.show()

# Display Confusion Matrix
cm_img = glob.glob('helmet_safety_run/yolov8n_helmet/confusion_matrix.png')
if cm_img:
    plt.figure(figsize=(10, 8))
    plt.imshow(mpimg.imread(cm_img[0]))
    plt.axis('off')
    plt.title('Class Confusion Matrix', fontsize=14)
    plt.show()

### Step 8: Validate Model on Test Set
Compute official Mean Average Precision (mAP@0.50 and mAP@0.50:0.95).

In [ ]:
# Validate the best trained weights
best_weights = 'helmet_safety_run/yolov8n_helmet/weights/best.pt'
trained_model = YOLO(best_weights)

metrics = trained_model.val(data='/content/dataset/data.yaml')

print(f"\n--- Validation Summary ---")
print(f"mAP@0.50:     {metrics.box.map50 * 100:.2f}%")
print(f"mAP@0.50:0.95:{metrics.box.map * 100:.2f}%")
print(f"Mean Precision: {metrics.box.mp * 100:.2f}%")
print(f"Mean Recall:    {metrics.box.mr * 100:.2f}%")

### Step 9: Test Inference on Sample Traffic Images

In [ ]:
# Run prediction on test images
test_results = trained_model.predict(
    source='/content/dataset/test/images',
    conf=0.40,
    iou=0.45,
    save=True,
    project='test_predictions',
    name='eval'
)

# Preview first 3 test prediction results
pred_images = glob.glob('test_predictions/eval/*.jpg')[:3]
for p_img in pred_images:
    plt.figure(figsize=(10, 6))
    plt.imshow(mpimg.imread(p_img))
    plt.axis('off')
    plt.show()

### Step 10: Download Trained Weights (`best.pt`)
Download the trained weights directly to your local laptop and save them into your project's `models/` directory as `helmet_yolov8n.pt`.

In [ ]:
from google.colab import files

# Download the best trained weights to your laptop
files.download('helmet_safety_run/yolov8n_helmet/weights/best.pt')